# PPThinning\n\nSimulating point processes by thinning.\n\nPython port of the MATLAB `PPThinning` helpfile (`helpfiles/PPThinning.m`).

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from nstat import Covariate, nspikeTrain, nstColl, CIF
from nstat.notebook_figures import FigureTracker
np.random.seed(0)
OUTPUT_ROOT = REPO_ROOT / "output" / "notebook_images"
__tracker = FigureTracker(topic="PPThinning", output_root=OUTPUT_ROOT, expected_count=3)
def capture(label, fn):
    fig = __tracker.new_figure(label); plt.close(fig); fn(); __tracker._active_fig = plt.gcf()


In [ ]:
delta = 0.001; Tmax = 100; time = np.arange(0, Tmax + delta, delta); f = 0.1
lambdaData = 10*np.sin(2*np.pi*f*time) + 10
lam = Covariate(time, lambdaData, "Lambda(t)", "time", "s", "Hz", ["lambda_1"])
lambdaBound = float(np.max(lambdaData))
rng = np.random.default_rng(0)
N = int(lambdaBound * 1.5 * Tmax)
u = rng.random(N); w = -np.log(u) / lambdaBound
tSpikes = np.cumsum(w); tSpikes = tSpikes[tSpikes <= Tmax]
ratio = np.asarray(lam.getValueAt(tSpikes)).flatten() / lambdaBound
u2 = rng.random(len(ratio)); tThin = tSpikes[ratio >= u2]
n1 = nspikeTrain(tSpikes); n2 = nspikeTrain(tThin)

__tracker.new_figure("constant-rate vs thinned process")
plt.subplot(2,2,1); n1.plot(); plt.xlim(0, Tmax/4); plt.title("constant-rate raster")
plt.subplot(2,2,2); plt.hist(np.diff(tSpikes), bins=60); plt.title("constant ISI hist")
plt.subplot(2,2,3); n2.plot(); plt.xlim(0, Tmax/4); plt.title("thinned raster")
plt.subplot(2,2,4); plt.hist(np.diff(tThin), bins=60); plt.title("thinned ISI hist")
plt.tight_layout()

__tracker.new_figure("thinned spikes + scaled rate")
n2.plot(); plt.xlim(0, Tmax/4)
plt.plot(time, lambdaData / lambdaBound, "b", linewidth=1.5)
plt.title("thinned spikes vs scaled rate p(t)")

spikeColl = CIF.simulateCIFByThinningFromLambda(lam, 20, seed=1)
capture("20 thinning realizations + rate", lambda: spikeColl.plot())
__tracker.finalize()
